# 01. Исследовательский анализ и подготовка данных

В первом ноутбуке решаются три задачи: оценка структуры и качества датасета, подготовка очищенной версии для последующего анализа, фиксация базовых наблюдений по выручке и сезонности.

После очистки в работе остаётся около 800 тысяч строк и приблизительно 5,8 тысяч уникальных клиентов. Этого объёма достаточно для построения когортного retention и проверки статистических гипотез на разумном уровне мощности.

## Импорт библиотек

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 50)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

## Загрузка датасета

Исходный файл содержит два листа, по одному на каждый год наблюдений. Объединяем их в единый датафрейм для сквозного анализа двух полных лет.

In [ ]:
DATA_PATH = '../data/online_retail_II.xlsx'

df_09_10 = pd.read_excel(DATA_PATH, sheet_name='Year 2009-2010')
df_10_11 = pd.read_excel(DATA_PATH, sheet_name='Year 2010-2011')

df = pd.concat([df_09_10, df_10_11], ignore_index=True)
print(f'Размер: {df.shape[0]:,} строк, {df.shape[1]} колонок')
df.head()

Описание полей:
- Invoice: номер чека. Префикс 'C' указывает на возврат и должен быть исключён из анализа.
- StockCode: уникальный артикул товара.
- Description: текстовое наименование позиции.
- Quantity: количество. Отрицательные значения соответствуют возвратам.
- InvoiceDate: дата и время совершения покупки.
- Price: цена единицы товара.
- Customer ID: идентификатор клиента. Ключевое поле для построения когортного анализа.
- Country: страна доставки.

## Оценка пропусков

Качество данных оцениваем по двум направлениям: общий уровень пропусков и наличие пропусков в Customer ID. Последнее критично, поскольку анонимные транзакции невозможно отнести к определённой когорте.

In [ ]:
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({'missing': missing, 'pct': missing_pct})

Около 22% строк не содержат Customer ID. Это могут быть гостевые покупки или технические записи. Для retention-аналитики такие строки бесполезны и подлежат удалению.

## Очистка данных

Применяемые правила:
1. Удаляются строки без Customer ID (необходимое условие для когортного анализа).
2. Удаляются возвраты, идентифицируемые по префиксу 'C' в номере чека.
3. Удаляются строки с неположительными значениями количества или цены (предположительно ошибки ввода и технические проводки).
4. Добавляются производные колонки Revenue и InvoiceMonth, востребованные во всех последующих ноутбуках.

In [ ]:
before = len(df)

df = df.dropna(subset=['Customer ID'])
df = df[~df['Invoice'].astype(str).str.startswith('C')]
df = df[(df['Quantity'] > 0) & (df['Price'] > 0)]
df['Customer ID'] = df['Customer ID'].astype(int)
df['Revenue'] = df['Quantity'] * df['Price']
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['InvoiceMonth'] = df['InvoiceDate'].dt.to_period('M').dt.to_timestamp()

print(f'Было строк: {before:,}')
print(f'Стало строк: {len(df):,}')
print(f'Удалено:    {before - len(df):,} ({(1 - len(df)/before)*100:.1f}%)')

Удалено около четверти исходных строк, преимущественно за счёт пропусков в Customer ID. Для розничного датасета это ожидаемая величина потерь, и оставшийся объём данных репрезентативен для целей анализа.

## Базовые характеристики выборки

In [ ]:
print(f'Период наблюдений:    {df["InvoiceDate"].min():%Y-%m-%d}  ->  {df["InvoiceDate"].max():%Y-%m-%d}')
print(f'Уникальных клиентов:  {df["Customer ID"].nunique():,}')
print(f'Уникальных заказов:   {df["Invoice"].nunique():,}')
print(f'Уникальных артикулов: {df["StockCode"].nunique():,}')
print(f'Общая выручка:        £{df["Revenue"].sum():,.0f}')
print(f'Стран в данных:       {df["Country"].nunique()}')

## Географическое распределение выручки

Структура выручки по странам определяет область применимости выводов. Если выручка концентрированная, итоги анализа справедливы прежде всего для доминирующего рынка.

In [ ]:
country_revenue = df.groupby('Country')['Revenue'].sum().sort_values(ascending=False).head(10)
ax = country_revenue.plot(kind='barh', figsize=(10, 5), color='#4C72B0')
ax.invert_yaxis()
ax.set_title('Топ-10 стран по выручке')
ax.set_xlabel('Выручка, £')
plt.tight_layout()
plt.show()

Великобритания формирует более 85% совокупной выручки. Дальнейший анализ ведётся по полному датасету, однако в выводах учитывается локальный характер источника: цифровые оценки не следует переносить на другие рынки без отдельной проверки.

## Сезонность выручки

Сезонные колебания учитываются при интерпретации когорт. Декабрьские когорты традиционно содержат значительную долю покупателей подарков, чьё поведение отличается от среднего.

In [ ]:
monthly = df.groupby('InvoiceMonth').agg(
    revenue=('Revenue', 'sum'),
    orders=('Invoice', 'nunique'),
    customers=('Customer ID', 'nunique')
).reset_index()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(monthly['InvoiceMonth'], monthly['revenue'], marker='o', color='#4C72B0')
ax.set_title('Выручка по месяцам')
ax.set_ylabel('Выручка, £')
ax.set_xlabel('Месяц')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Зафиксирован выраженный пик выручки в ноябре-декабре в каждом из двух наблюдаемых лет. Это соответствует предновогоднему периоду и является поводом отдельно рассмотреть декабрьские когорты в ноутбуке 02.

## Распределение размера чека

Распределение чеков задаёт условия для статистических процедур, применяемых далее. В частности, скошенность распределения влияет на выбор между средним и медианой при сравнении групп.

In [ ]:
invoice_totals = df.groupby('Invoice')['Revenue'].sum()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(invoice_totals, bins=80, color='#4C72B0')
axes[0].set_title('Размер чека (полное распределение)')
axes[0].set_xlabel('£')
axes[0].set_ylabel('Количество чеков')

p99 = invoice_totals.quantile(0.99)
axes[1].hist(invoice_totals[invoice_totals < p99], bins=80, color='#55A868')
axes[1].set_title(f'Размер чека (без верхнего 1%, < £{p99:.0f})')
axes[1].set_xlabel('£')
plt.tight_layout()
plt.show()

print(f'Медианный чек: £{invoice_totals.median():.2f}')
print(f'Средний чек:   £{invoice_totals.mean():.2f}')

Распределение характеризуется выраженной правосторонней скошенностью: большинство чеков сосредоточены в области малых значений, а длинный правый хвост формируется оптовыми клиентами. Превышение среднего над медианой почти в два раза подтверждает наличие тяжёлого хвоста.

Методологическое следствие: при сравнении средних значений будут параллельно приведены медианы, чтобы исключить интерпретацию, опирающуюся исключительно на хвост распределения. В качестве основного теста используется t-критерий Уэлча, устойчивый к различию дисперсий.

## Сохранение очищенной выборки

Очищенный датафрейм сохраняется в формате parquet для использования в последующих ноутбуках без повторной обработки.

In [ ]:
df.to_parquet('../data/clean.parquet', index=False)
print('Сохранено: data/clean.parquet')

## Резюме

Подготовлен датасет, пригодный для retention-аналитики и статистической проверки гипотез: два года наблюдений, около 5,8 тысяч уникальных клиентов после очистки, выраженная декабрьская сезонность, скошенное распределение чеков. Доминирование британского рынка ограничивает применимость числовых оценок одной географией, но не влияет на методологию.

В ноутбуке 02 строится когортный retention и проверяется гипотеза о связи между размером первого чека и вероятностью возврата клиента.